This is a starter notebook for an updated module 5 of ML Zoomcamp

The code is based on the modules 3 and 4. We use the same dataset: [telco customer churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

# PART 1 - IDE SETUP
## ENABLE CODE COMPLETION IN JUPYTER LABS / JUPYTER NOTEBOOKS
 - Open Settings → Settings Editor. 
 - Select Code Completion. 
 - Turn on Enable autocompletion.

# PART 2 - PYTHON MACHINE LEARNING SETUP

## 2.1 - IMPORT PACKAGES 

In [5]:
import pandas as pd   # USED TO CREATE DATAFRAME OBJECT
import numpy as np    # USED TO ACCELERATE MATH OPERATIONS
import sklearn        # USED TO PREPARE DATA FOR MODELING, AND CREATE AND USE A MODEL
import pickle         # USED TO CONVERT PYTHON OBJECTS INTO BINARIES

In [6]:
print(f'pandas=={pd.__version__}')
print(f'numpy=={np.__version__}')
print(f'sklearn=={sklearn.__version__}')

pandas==3.0.5
numpy==2.4.6
sklearn==1.9.1


In [7]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

##   2.2 - GET RAW DATA AND PREPARE THEM FOR MODELING 

In [8]:
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_url)

df.columns = df.columns.str.lower().str.replace(' ', '_')

# categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)  # REMOVED BY CT
categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()  # ADDED BY CT

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

##   2.3 - CREATE TARGET LABELS DATAFRAME

In [9]:
y_train = df.churn

##   2.4 - FEATURE ENGINEERING - SPECIFY FEATURES TO INCLUDE IN THE MODEL
  - CREATE A LIST OF NUMERICAL FEATURES
  - CREATE A LIST OF CATEGORICAL FEATURES

In [10]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

## 2.5 - STEPS SKIPPED IN THE LESSON FOR EXPEDIENCY PURPOSES
THE LESSON SKIPPED THE STEPS INVOLVED IN PARTITIONING THE DATASET
 - THERE IS NO TRAIN, VALIDATION, OR TEST DATASET
 - THERE ARE NO K-FOLD SPLITS TO INCREASE THE TRAINING ACCURACY FOR SMALL DATASETS

## 2.6 - CREATE THE MODEL

In [11]:
# DictVectorizer()
#   - CONVERTS A DATAFRAME INTO A LIST OF DICTIONARIES
#   - IMPLEMENTS ONE-HOT ENCODING FOR EACH DATAPOINT IN THE DATASET
dv = DictVectorizer()

# SET REGULARIZATION VALUE
C = 1.0

# CREATE THE TRAINING DICTIONARY USING THE ENTIRE DATASET
#   IN PRACTICE, THIS IS A HORRIBLE IDEA AS IT BECOMES IMPOSSIBLE TO EVALUATE A MODEL WITHOUT 
#   A VALIDATION OR TEST DATASET
train_dict = df[categorical + numerical].to_dict(orient='records')

# USE TRAINING DATASET TO PREPARE FOR CREATING THE MODEL 
#   - CALCULATE AND SET THE WEIGHTING BIAS
#   - CALCULATE AND SET THE FEATURE WEIGHTS
X_train = dv.fit_transform(train_dict)

# BUG FIX: 
#    THE 'LogisticRegression' FUNCTION 'liblinear' MODEL ONLY WORKS WITH SPARSE MARTICIES
#    CONVERT 64-BIT 'DENSE MATRIX' INDEX VALUES TO 32-BIT 'SPARSE MATRIX' INDEX VALUES 
X_train.indices = X_train.indices.astype("int32")
X_train.indptr  = X_train.indptr.astype("int32")


# CREATE THE MODEL OBJECT
#    SOLVER:               "liblinear"
#    REGULARIZATION VALUE: 1.0
model           = LogisticRegression(solver="liblinear", C=C)

# FIT THE MODEL OBJECT BASED UPON THE TRAINING FEATURES AND TARGET LABELS
model.fit(X_train, y_train)

,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some pen

## 2.7 - CREATE A PREDICTION AS WE HAVE ALWAYS DONE

In [ ]:
# CREATE THE DATA POINT USED IN THE TRAINING VIDEO
customer = {
    'gender': 'male',
    'seniorcitizen': 0,
    'partner': 'no',
    'dependents': 'yes',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 6,
    'monthlycharges': 29.85,
    'totalcharges': 129.85
}

# APPLY THE SAME DictVectorizer THAT WAS USED TO CREATE THE MODEL TO THE TEST DATAPOINT
#    THIS CONVERTS THE DATAPOINT INTO A SPARSE MATRIX WITH THE SAME SHAPE AS THE TRAINING DATASET

X = dv.transform([customer])

# DEBUG PRINT LINE
# print(dv.get_feature_names_out())
# print(X.toarray())

# MAKE THE PREDICTION
churn = model.predict_proba(X)[0,1]

# DEBUG
# print(f"churn:  {churn}")

# DECISION
if churn >- 0.5:
    print('send email with promo')
else:
    print('don''t do anything')

# PART 3: CREATE A BINARY FILE TO HOLD THE MACHINE LEARNING MODEL
  - BINARY FILE SHOULD HOLD BOTH
    - 'DictVectorizer' AND
    - 'model'

## 3.1 - CREATE THE BINARY FILE TO STORE THE TWO OBJECTS 'sklearn' REQUIRES FOR PREDICTIONS

In [ ]:
# CODE BLOCK USES pickle TO CONVERT A PYTHON OBJECT INTO A BINARY FILE
#    HERE WE ARE writing bytes (wb) TO A FILE NAMED "model.bin' 
#    THE BINARY FILE WILL CONTAIN THE 'DictVectorizer' AND THE 'model'
#    FILE WILL BE WRITTEN TO THE SAME DIRECTORY WHERE THIS Jupyter Notebook IS SAVED

# import pickle     # THE BELOW CODE WILL NOT WORK WITHOUT THE 'pickle' PACKAGE

with open('model.bin', 'wb') as f_out:
    pickle.dump((dv, model), f_out)

## 3.2 - VALIDATE THE BINARY FILE

In [ ]:
# VERIFY THAT 'model.bin' WORKS
#   1. RESTART KERNEL AND CLEAR ALL OUTPUT TO FORGET dv
#   2. IMPORT pickle SINCE IT WAS FORGETTEN WHEN THE KERNEL RESTARTED
#   3. RUN THIS CODE TO RESTORE THE VECTORIZER
import pickle

with open('model.bin', 'rb') as f_in:
    (dv, model) = pickle.load(f_in)

# DISPLAY THE DISK VECTORIZER METADATA
dv

# PART 4: USE THE BINARY FILE TO MAKE PREDICTIONS
  - USE THE BINARY TO MAKE A PREDICTION

## 4.1 - USE THE BINARY FILE TO DUPLICATE LESSON PREDICTION

In [ ]:
# 4.1.1 - RETREIVE THE BINARIES
import pickle

with open('model.bin', 'rb') as f_in:
    (dv, model) = pickle.load(f_in)
    
# 4.1.2 - CREATE THE DATA POINT USED IN THE TRAINING VIDEO
customer = {
    'gender': 'male',
    'seniorcitizen': 0,
    'partner': 'no',
    'dependents': 'yes',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 6,
    'monthlycharges': 29.85,
    'totalcharges': 129.85
}

# 4.1.3 - APPLY THE SAME DictVectorizer THAT WAS USED TO CREATE THE MODEL TO THE TEST DATAPOINT
#    THIS CONVERTS THE DATAPOINT INTO A SPARSE MATRIX WITH THE SAME SHAPE AS THE TRAINING DATASET

X = dv.transform([customer])

# DEBUG PRINT LINE
# print(dv.get_feature_names_out())
# print(X.toarray())

# 4.1.4 - MAKE THE PREDICTION
churn = model.predict_proba(X)[0,1]

# DEBUG
# print(f"churn:  {churn}")

# 4.1.5 - DECISION
if churn >- 0.5:
    print('send email with promo')
else:
    print('don''t do anything')

## 4.2 - USE THE BINARY FILE TO MAKE A PREDICTION

In [2]:
# 4.2.1 - RETREIVE THE BINARIES (IF THEY WERE NOT RETRIEVED IN 4.1.1)
import pickle

with open('model.bin', 'rb') as f_in:
    (dv, model) = pickle.load(f_in)
    
# 4.2.2 - CREATE THE DATA POINT TO USE WHEN MAKING THE PREDICTION
datapoint = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}


# 4.2.3 - APPLY THE SAME DictVectorizer THAT WAS USED TO CREATE THE MODEL TO THE TEST DATAPOINT
#    THIS CONVERTS THE DATAPOINT INTO A SPARSE MATRIX WITH THE SAME SHAPE AS THE TRAINING DATASET

X = dv.transform([datapoint])

# DEBUG
# print(dv.get_feature_names_out())
# print(X.toarray())

# 4.1.4 - MAKE THE PREDICTION
churn = model.predict_proba(X)[0,1]

# DEBUG
# print(f"\r\nchurn:  {churn}\r\r")

# 4.2.5 - DECISION
if churn >- 0.5:
    print('send email with promo')
else:
    print('don''t do anything')

send email with promo


# PART 5: PIPELINES
  SIMPLIFY SECTION 2.6 CODE WITH A PIPELINE

In [15]:
# IMPORT PACKAGES AND FUNCTIONS NECESSARY FOR PART 5
#    REMEMBER THAT ALL IMPORTED PACKAGES WERE DROPPED WHEN WE RESTARTED THE KERNEL
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    # DICTIONARY VECTORIZER
    DictVectorizer(),
    # LogisticRegression(solver='liblinear')
    LogisticRegression(solver='lbfgs', max_iter=5000)
)

# SEPARATE STEPS CAN BE COMBINED WITH A PIPELINE
#   fitting X_train to the model
#   fitting y_train to the model
pipeline.fit(train_dict, y_train)

#################################################################
# REFERENCING TWO OBJECTS WHEN CREATING THE BINARY
# with open('model.bin', 'wb') as f_out:
#    pickle.dump((dv, model), f_out)

with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

#################################################################
# REFERENCING TWO OBJECTS WHEN USING THE BINARY
# with open('model.bin', 'rb') as f_in:
#     (dv, model) = pickle.load(f_in)
with open('model.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)
    
# MAKING A PREDICTION CAN NOW BE DONE IN ONE STEP
churn = pipeline.predict_proba(customer)[[0,1]

print

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",True
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


# PART 6 - CONVERT JUPYTER NOTEBOOK INTO A PYTHON SCRIPT
1. Open a terminal
2. Run the following command \
       jupyter nbconvert --to=script workshop-uv-fastapi.ipynb
3. Rename to 'train.py'